# 02. 커스텀 장애물 회피 환경 + 5개 센서 통합

**모듈**: M3 - 시뮬레이션 환경
**날짜**: 2026-03-09

최종 프로젝트에 사용할 **ObstacleAvoidanceEnv**를 만듭니다.

```
env.step(action)
  → observation: 5개 센서 dict (camera, distance, audio, imu, llm)
  → reward: 목표 도달(+100), 충돌(-50), 스텝(-1)
  → info: LLM 상황 해석 텍스트
```

**실습 2(환경) + 실습 3(센서 통합)을 하나로 합침**

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import json

mpl.rcParams['font.family'] = 'AppleGothic'
mpl.rcParams['axes.unicode_minus'] = False
np.random.seed(42)
print('라이브러리 로드 완료!')

## 1. ObstacleAvoidanceEnv 구현

### 환경 설정
- **맵**: NxN 연속 공간 (정수 격자가 아닌 실수 좌표)
- **에이전트**: (x, y) 위치 + 방향(angle)
- **장애물**: 원형 (중심 + 반지름)
- **목표**: 도달할 위치
- **행동**: 4방향 (상/하/좌/우) 또는 8방향

### 센서 통합
- 카메라: 에이전트 시점 64x64 RGB
- 거리: 8방향 레이캐스팅
- 오디오: 목표/장애물 방향 스테레오
- IMU: 속도, 가속도, 방향
- LLM: 규칙 기반 상황 해석 텍스트

In [ ]:
class ObstacleAvoidanceEnv(gym.Env):
    """5개 센서를 가진 장애물 회피 환경"""

    metadata = {'render_modes': ['human']}
    ACTION_NAMES = ['상(N)', '우상(NE)', '우(E)', '우하(SE)', '하(S)', '좌하(SW)', '좌(W)', '좌상(NW)']

    def __init__(self, config=None):
        super().__init__()

        # 기본 설정
        cfg = config or {}
        self.map_size = cfg.get('map_size', 10.0)
        self.step_size = cfg.get('step_size', 0.5)
        self.max_steps = cfg.get('max_steps', 100)
        self.obstacle_radius = cfg.get('obstacle_radius', 0.8)

        # 장애물 설정
        self.obstacle_configs = cfg.get('obstacles', [
            {'pos': [3.0, 3.0]},
            {'pos': [5.0, 6.0]},
            {'pos': [7.0, 4.0]},
        ])

        self.start_pos = np.array(cfg.get('start', [1.0, 1.0]), dtype=np.float32)
        self.goal_pos = np.array(cfg.get('goal', [9.0, 9.0]), dtype=np.float32)

        # 행동: 8방향 이동
        self.action_space = spaces.Discrete(8)

        # 관측: 5개 센서 (LLM은 info에)
        self.observation_space = spaces.Dict({
            'camera': spaces.Box(0, 1, shape=(3, 64, 64), dtype=np.float32),
            'distance': spaces.Box(0, 1, shape=(8,), dtype=np.float32),
            'audio': spaces.Box(-1, 1, shape=(2, 50), dtype=np.float32),
            'imu': spaces.Box(-10, 10, shape=(6,), dtype=np.float32),
        })

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_pos = self.start_pos.copy()
        self.agent_angle = 0.0  # 라디안 (0=동쪽)
        self.prev_pos = self.agent_pos.copy()
        self.velocity = np.zeros(2, dtype=np.float32)
        self.steps = 0

        # 장애물 위치 설정
        self.obstacles = [np.array(o['pos'], dtype=np.float32) for o in self.obstacle_configs]

        obs = self._get_observation()
        info = self._get_info()
        return obs, info

    def step(self, action):
        self.prev_pos = self.agent_pos.copy()

        # 8방향 이동
        angles = np.linspace(np.pi/2, np.pi/2 - 2*np.pi, 8, endpoint=False)  # N부터 시계방향
        dx = np.cos(angles[action]) * self.step_size
        dy = np.sin(angles[action]) * self.step_size

        new_pos = self.agent_pos + np.array([dx, dy], dtype=np.float32)
        new_pos = np.clip(new_pos, 0, self.map_size)

        # 충돌 체크
        collision = False
        for obs_pos in self.obstacles:
            if np.linalg.norm(new_pos - obs_pos) < self.obstacle_radius:
                collision = True
                break

        if not collision:
            self.velocity = new_pos - self.agent_pos
            self.agent_pos = new_pos
            self.agent_angle = angles[action]
        else:
            self.velocity = np.zeros(2, dtype=np.float32)

        self.steps += 1

        # 보상
        dist_to_goal = np.linalg.norm(self.agent_pos - self.goal_pos)
        prev_dist = np.linalg.norm(self.prev_pos - self.goal_pos)

        if dist_to_goal < 0.5:
            reward = 100.0
            terminated = True
        elif collision:
            reward = -50.0
            terminated = True
        else:
            # 목표에 가까워지면 보상, 멀어지면 패널티
            reward = (prev_dist - dist_to_goal) * 10.0 - 0.5
            terminated = False

        truncated = self.steps >= self.max_steps

        obs = self._get_observation()
        info = self._get_info()
        info['collision'] = collision
        info['dist_to_goal'] = dist_to_goal

        return obs, reward, terminated, truncated, info

    # ============ 센서 시뮬레이션 ============

    def _get_observation(self):
        return {
            'camera': self._sense_camera(),
            'distance': self._sense_distance(),
            'audio': self._sense_audio(),
            'imu': self._sense_imu(),
        }

    def _sense_camera(self):
        """64x64 RGB 카메라 이미지 생성"""
        img = np.zeros((64, 64, 3), dtype=np.float32)

        # 하늘 (위쪽 절반)
        img[:32, :, 2] = 0.7  # 파랑
        img[:32, :, 1] = 0.8
        # 바닥 (아래쪽 절반)
        img[32:, :, 1] = 0.6  # 초록
        img[32:, :, 0] = 0.2

        # 각 장애물을 시야에 렌더링
        for obs_pos in self.obstacles:
            diff = obs_pos - self.agent_pos
            dist = np.linalg.norm(diff)

            if dist < 5.0:  # 5m 이내만 보임
                # 장애물의 상대 각도
                angle = np.arctan2(diff[1], diff[0])
                rel_angle = angle - self.agent_angle

                # 시야각 ±60도 내에 있으면 렌더링
                if abs(rel_angle) < np.pi / 3:
                    # 화면상 x 위치 (중앙 = 32)
                    screen_x = int(32 + rel_angle * 32 / (np.pi / 3))
                    # 거리에 따른 크기
                    size = max(2, int(15 / max(dist, 0.5)))

                    x1 = max(0, screen_x - size)
                    x2 = min(64, screen_x + size)
                    y1 = max(0, 32 - size * 2)
                    y2 = min(64, 32 + size * 2)

                    img[y1:y2, x1:x2, 0] = 1.0  # 빨간 장애물
                    img[y1:y2, x1:x2, 1] = 0.0
                    img[y1:y2, x1:x2, 2] = 0.0

        # 목표 렌더링 (초록 사각형)
        diff_goal = self.goal_pos - self.agent_pos
        dist_goal = np.linalg.norm(diff_goal)
        if dist_goal < 6.0:
            angle_goal = np.arctan2(diff_goal[1], diff_goal[0])
            rel_angle_goal = angle_goal - self.agent_angle
            if abs(rel_angle_goal) < np.pi / 3:
                sx = int(32 + rel_angle_goal * 32 / (np.pi / 3))
                sz = max(2, int(10 / max(dist_goal, 0.5)))
                x1, x2 = max(0, sx-sz), min(64, sx+sz)
                y1, y2 = max(0, 32-sz), min(64, 32+sz)
                img[y1:y2, x1:x2, 0] = 0.0
                img[y1:y2, x1:x2, 1] = 1.0  # 초록 목표
                img[y1:y2, x1:x2, 2] = 0.0

        # 노이즈
        img += np.random.normal(0, 0.02, img.shape).astype(np.float32)
        img = np.clip(img, 0, 1)

        # (H,W,C) → (C,H,W)
        return img.transpose(2, 0, 1)

    def _sense_distance(self):
        """8방향 레이캐스팅 거리센서"""
        distances = np.ones(8, dtype=np.float32)
        ray_angles = np.linspace(0, 2*np.pi, 8, endpoint=False) + self.agent_angle
        max_range = 5.0

        for i, ray_angle in enumerate(ray_angles):
            # 레이 방향
            ray_dir = np.array([np.cos(ray_angle), np.sin(ray_angle)])

            min_dist = max_range

            # 장애물과의 교차점
            for obs_pos in self.obstacles:
                # 레이-원 교차 테스트
                to_obs = obs_pos - self.agent_pos
                proj = np.dot(to_obs, ray_dir)
                if proj > 0:  # 전방에 있을 때만
                    perp_dist = abs(np.cross(ray_dir, to_obs))
                    if perp_dist < self.obstacle_radius:
                        hit_dist = proj - np.sqrt(max(0, self.obstacle_radius**2 - perp_dist**2))
                        if 0 < hit_dist < min_dist:
                            min_dist = hit_dist

            # 벽과의 거리
            for axis in [0, 1]:
                if ray_dir[axis] > 0.001:
                    wall_dist = (self.map_size - self.agent_pos[axis]) / ray_dir[axis]
                    min_dist = min(min_dist, wall_dist)
                elif ray_dir[axis] < -0.001:
                    wall_dist = -self.agent_pos[axis] / ray_dir[axis]
                    min_dist = min(min_dist, wall_dist)

            distances[i] = min(min_dist / max_range, 1.0)  # 0~1 정규화

        return distances

    def _sense_audio(self):
        """스테레오 오디오: 가장 가까운 장애물/목표 방향 소리"""
        n_samples = 50
        left = np.random.normal(0, 0.05, n_samples).astype(np.float32)
        right = np.random.normal(0, 0.05, n_samples).astype(np.float32)

        # 가장 가까운 장애물의 소리
        if self.obstacles:
            dists = [np.linalg.norm(o - self.agent_pos) for o in self.obstacles]
            closest_idx = np.argmin(dists)
            closest_obs = self.obstacles[closest_idx]
            closest_dist = dists[closest_idx]

            if closest_dist < 3.0:  # 3m 이내에서 소리 들림
                intensity = (3.0 - closest_dist) / 3.0 * 0.5
                t = np.linspace(0, 4*np.pi, n_samples)
                signal = intensity * np.sin(t).astype(np.float32)

                # 방향에 따라 좌/우 배분
                diff = closest_obs - self.agent_pos
                angle = np.arctan2(diff[1], diff[0]) - self.agent_angle

                left += signal * max(0, -np.sin(angle))   # 왼쪽에 있으면 왼쪽 채널
                right += signal * max(0, np.sin(angle))   # 오른쪽이면 오른쪽 채널

        return np.stack([np.clip(left, -1, 1), np.clip(right, -1, 1)])

    def _sense_imu(self):
        """IMU: 속도, 가속도, 회전"""
        speed = np.linalg.norm(self.velocity)
        noise = np.random.normal(0, 0.01, 6).astype(np.float32)

        return np.array([
            self.velocity[0],                      # vx
            self.velocity[1],                      # vy
            0.0,                                   # ax (간소화)
            0.0,                                   # ay
            self.agent_angle,                      # angular_velocity (간소화)
            np.degrees(self.agent_angle) % 360,    # angle (도)
        ], dtype=np.float32) + noise

    # ============ LLM 센서 (info에 포함) ============

    def _get_info(self):
        """LLM 상황 해석을 info에 포함"""
        distances = self._sense_distance()
        dir_names = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']

        # 상황 해석
        parts = []

        # 가장 가까운 장애물
        closest_idx = np.argmin(distances)
        closest_val = distances[closest_idx]
        if closest_val < 0.3:
            parts.append(f'{dir_names[closest_idx]}방향 매우 가까운 장애물 (위험!)')
        elif closest_val < 0.6:
            parts.append(f'{dir_names[closest_idx]}방향 장애물 감지')
        else:
            parts.append('주변 안전')

        # 목표까지 거리/방향
        diff_goal = self.goal_pos - self.agent_pos
        dist_goal = np.linalg.norm(diff_goal)
        angle_goal = np.arctan2(diff_goal[1], diff_goal[0]) - self.agent_angle
        if angle_goal > np.pi / 4:
            parts.append(f'목표: 왼쪽 {dist_goal:.1f}m')
        elif angle_goal < -np.pi / 4:
            parts.append(f'목표: 오른쪽 {dist_goal:.1f}m')
        else:
            parts.append(f'목표: 정면 {dist_goal:.1f}m')

        # 속도 상태
        speed = np.linalg.norm(self.velocity)
        if speed > 0.3:
            parts.append('이동 중')
        else:
            parts.append('정지/저속')

        return {
            'llm_interpretation': '. '.join(parts) + '.',
            'agent_pos': self.agent_pos.copy(),
            'agent_angle': self.agent_angle,
            'steps': self.steps,
        }

print('ObstacleAvoidanceEnv 정의 완료!')

## 2. 환경 테스트

In [ ]:
# 기본 환경 생성
env = ObstacleAvoidanceEnv()
obs, info = env.reset(seed=42)

print('=== 초기 상태 ===')
print(f'에이전트 위치: {info["agent_pos"]}')
print(f'목표 위치: {env.goal_pos}')
print(f'장애물: {[o.tolist() for o in env.obstacles]}')
print()

# 센서 데이터 확인
print('=== 5개 센서 데이터 ===')
for name, data in obs.items():
    print(f'  {name:>10}: shape={data.shape}, range=[{data.min():.2f}, {data.max():.2f}]')
print(f'  {"llm":>10}: "{info["llm_interpretation"]}"')

# 한 스텝 실행
print(f'\n--- 행동: 우상(NE) ---')
obs, reward, term, trunc, info = env.step(1)  # NE
print(f'  새 위치: {info["agent_pos"]}')
print(f'  보상: {reward:.2f}')
print(f'  LLM: "{info["llm_interpretation"]}"')

In [ ]:
# 센서 대시보드 함수
def show_sensor_dashboard(obs, info, title='센서 대시보드'):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # 1. 카메라 (C,H,W) → (H,W,C)
    axes[0, 0].imshow(obs['camera'].transpose(1, 2, 0))
    axes[0, 0].set_title('[CAM] 카메라 (64x64 RGB)')
    axes[0, 0].axis('off')

    # 2. 거리센서 (레이더)
    ax_radar = fig.add_subplot(2, 2, 2, polar=True)
    dirs = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    angles = np.linspace(0, 2*np.pi, 8, endpoint=False).tolist() + [0]
    values = obs['distance'].tolist() + [obs['distance'][0]]
    ax_radar.plot(angles, values, 'o-', linewidth=2, color='blue')
    ax_radar.fill(angles, values, alpha=0.25, color='blue')
    ax_radar.set_thetagrids([i*45 for i in range(8)], dirs)
    ax_radar.set_ylim(0, 1.2)
    ax_radar.set_title('[DIST] 거리센서', pad=20)
    axes[0, 1].set_visible(False)

    # 3. 오디오
    axes[1, 0].plot(obs['audio'][0], color='blue', alpha=0.7, label='Left')
    axes[1, 0].plot(obs['audio'][1], color='red', alpha=0.7, label='Right')
    axes[1, 0].set_title('[MIC] 오디오')
    axes[1, 0].legend()
    axes[1, 0].set_ylim(-0.8, 0.8)

    # 4. IMU
    imu_labels = ['vx', 'vy', 'ax', 'ay', 'ang_v', 'angle']
    colors = ['#2196F3', '#2196F3', '#FF9800', '#FF9800', '#4CAF50', '#4CAF50']
    axes[1, 1].barh(imu_labels, obs['imu'], color=colors)
    axes[1, 1].set_title('[IMU] 관성센서')

    # LLM
    fig.text(0.5, 0.02,
             f'[LLM] {info["llm_interpretation"]}',
             ha='center', fontsize=10, style='italic',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    return fig

# 현재 상태 대시보드
fig = show_sensor_dashboard(obs, info, '환경 시작 직후')
plt.show()

## 3. 다양한 맵 설정

JSON으로 환경 설정을 파라미터화합니다.

In [ ]:
# 3가지 환경 설정
configs = {
    'simple': {
        'map_size': 10.0,
        'obstacles': [{'pos': [5.0, 5.0]}],
        'start': [1.0, 1.0],
        'goal': [9.0, 9.0],
        'max_steps': 80,
    },
    'medium': {
        'map_size': 10.0,
        'obstacles': [
            {'pos': [3.0, 3.0]},
            {'pos': [5.0, 6.0]},
            {'pos': [7.0, 4.0]},
        ],
        'start': [1.0, 1.0],
        'goal': [9.0, 9.0],
        'max_steps': 100,
    },
    'complex': {
        'map_size': 12.0,
        'obstacles': [
            {'pos': [3.0, 2.0]}, {'pos': [3.0, 5.0]}, {'pos': [3.0, 8.0]},
            {'pos': [6.0, 3.0]}, {'pos': [6.0, 6.0]}, {'pos': [6.0, 9.0]},
            {'pos': [9.0, 4.0]}, {'pos': [9.0, 7.0]},
        ],
        'start': [1.0, 1.0],
        'goal': [11.0, 11.0],
        'max_steps': 150,
        'obstacle_radius': 0.7,
    },
}

# JSON으로 저장
for name, cfg in configs.items():
    with open(f'configs/env_{name}.json', 'w') as f:
        json.dump(cfg, f, indent=2)
    print(f'configs/env_{name}.json 저장 완료')

# 맵 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, cfg) in zip(axes, configs.items()):
    ms = cfg['map_size']
    ax.set_xlim(0, ms)
    ax.set_ylim(0, ms)
    ax.set_aspect('equal')

    # 장애물
    r = cfg.get('obstacle_radius', 0.8)
    for obs in cfg['obstacles']:
        circle = plt.Circle(obs['pos'], r, color='red', alpha=0.5)
        ax.add_patch(circle)

    # 시작/목표
    ax.plot(*cfg['start'], 'bs', markersize=12, label='시작')
    ax.plot(*cfg['goal'], 'g*', markersize=15, label='목표')

    ax.set_title(f'{name} ({len(cfg["obstacles"])}개 장애물)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('3가지 환경 설정', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 에피소드 실행 + 경로 시각화

In [ ]:
def run_episode(env, agent_fn=None, seed=42):
    """한 에피소드 실행, 경로 기록"""
    obs, info = env.reset(seed=seed)
    path = [info['agent_pos'].copy()]
    total_reward = 0
    llm_logs = [info['llm_interpretation']]

    while True:
        if agent_fn:
            action = agent_fn(obs, info)
        else:
            action = env.action_space.sample()

        obs, reward, term, trunc, info = env.step(action)
        path.append(info['agent_pos'].copy())
        total_reward += reward
        llm_logs.append(info['llm_interpretation'])

        if term or trunc:
            break

    return path, total_reward, llm_logs, info

def visualize_episode(env, path, title='에피소드'):
    """경로 시각화"""
    fig, ax = plt.subplots(figsize=(7, 7))
    ms = env.map_size

    # 장애물
    for obs_pos in env.obstacles:
        circle = plt.Circle(obs_pos, env.obstacle_radius, color='red', alpha=0.4)
        ax.add_patch(circle)
        ax.text(obs_pos[0], obs_pos[1], 'X', ha='center', va='center', fontsize=12, color='darkred')

    # 목표
    ax.plot(*env.goal_pos, 'g*', markersize=20, label='목표')

    # 경로
    path_arr = np.array(path)
    ax.plot(path_arr[:, 0], path_arr[:, 1], 'b-', alpha=0.5, linewidth=1.5)
    ax.plot(path_arr[0, 0], path_arr[0, 1], 'bs', markersize=10, label='시작')
    ax.plot(path_arr[-1, 0], path_arr[-1, 1], 'bo', markersize=10, label='종료')

    ax.set_xlim(0, ms)
    ax.set_ylim(0, ms)
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    return fig

# 3가지 맵에서 랜덤 에이전트 실행
fig_all, axes_all = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, cfg) in zip(axes_all, configs.items()):
    env = ObstacleAvoidanceEnv(config=cfg)
    path, reward, llm_logs, info = run_episode(env)

    ms = cfg['map_size']
    r = cfg.get('obstacle_radius', 0.8)

    for obs_pos in env.obstacles:
        circle = plt.Circle(obs_pos, r, color='red', alpha=0.4)
        ax.add_patch(circle)

    ax.plot(*env.goal_pos, 'g*', markersize=15)
    path_arr = np.array(path)
    ax.plot(path_arr[:, 0], path_arr[:, 1], 'b-', alpha=0.5)
    ax.plot(path_arr[0, 0], path_arr[0, 1], 'bs', markersize=8)
    ax.plot(path_arr[-1, 0], path_arr[-1, 1], 'bo', markersize=8)

    ax.set_xlim(0, ms)
    ax.set_ylim(0, ms)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{name}: {len(path)}스텝, 보상={reward:.0f}')

plt.suptitle('랜덤 에이전트: 3가지 맵', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 규칙 기반 에이전트 (거리센서 활용)

"가장 안전한 방향으로 이동 + 목표 방향 가중"하는 간단한 에이전트.

In [ ]:
def rule_based_agent(obs, info):
    """거리센서 기반 규칙 에이전트"""
    distances = obs['distance']  # (8,) 각 방향 거리

    # 목표 방향 계산
    agent_pos = info['agent_pos']
    goal_diff = np.array([9.0, 9.0]) - agent_pos  # 기본 목표
    goal_angle = np.arctan2(goal_diff[1], goal_diff[0])
    agent_angle = info['agent_angle']

    # 8방향 점수: 거리 멀수록 안전(높은 점수) + 목표 방향 보너스
    direction_angles = np.linspace(np.pi/2, np.pi/2 - 2*np.pi, 8, endpoint=False)

    scores = distances.copy()  # 안전도

    # 목표 방향 보너스
    for i, da in enumerate(direction_angles):
        angle_diff = abs(da - goal_angle)
        angle_diff = min(angle_diff, 2*np.pi - angle_diff)
        goal_bonus = max(0, 1 - angle_diff / np.pi) * 0.3
        scores[i] += goal_bonus

    return int(np.argmax(scores))

# medium 맵에서 비교
env = ObstacleAvoidanceEnv(config=configs['medium'])

# 랜덤 vs 규칙 에이전트
n_tests = 20
random_rewards = []
rule_rewards = []

for seed in range(n_tests):
    _, r_random, _, _ = run_episode(env, agent_fn=None, seed=seed)
    _, r_rule, _, _ = run_episode(env, agent_fn=rule_based_agent, seed=seed)
    random_rewards.append(r_random)
    rule_rewards.append(r_rule)

print(f'medium 맵 {n_tests} 에피소드:')
print(f'  랜덤: 평균 {np.mean(random_rewards):.1f}, 최대 {np.max(random_rewards):.0f}')
print(f'  규칙: 평균 {np.mean(rule_rewards):.1f}, 최대 {np.max(rule_rewards):.0f}')

# 규칙 에이전트 경로
path_rule, reward_rule, llm_logs, info = run_episode(env, agent_fn=rule_based_agent, seed=0)
fig = visualize_episode(env, path_rule, f'규칙 에이전트: {len(path_rule)}스텝, 보상={reward_rule:.0f}')
plt.show()

# LLM 해석 로그 (처음 5개)
print(f'\n--- LLM 상황 해석 (처음 5스텝) ---')
for i, log in enumerate(llm_logs[:5]):
    print(f'  Step {i}: {log}')

## 6. 파이프라인 연결

M1~M3가 완성한 것과 앞으로 할 것:

In [ ]:
print('=== M1~M3 완성 + 앞으로의 파이프라인 ===')
print()
print('[M1] SensorManager        → 5개 센서 클래스 (독립 시뮬레이터)')
print('[M2] CameraEncoder         → CNN으로 이미지→16차원 벡터')
print('[M3] ObstacleAvoidanceEnv  → Gymnasium 환경 + 5개 센서 통합  ← 완료!')
print()
print('env.step(action) 한 번 호출하면:')
print('  camera:   (3,64,64) ─→ M5: VLM Encoder ─→ (16,)')
print('  distance: (8,)      ─→ M5: Dist Encoder ─→ (16,)')
print('  audio:    (2,50)    ─→ M5: Audio Encoder ─→ (16,)')
print('  imu:      (6,)      ─→ M5: IMU Encoder ──→ (16,)')
print('  llm:      text      ─→ M5: Lang Encoder ─→ (16,)')
print('                               │')
print('                        M6: Fusion (5x16 → 64)')
print('                               │')
print('                        M7: V-JEPA (미래 예측)')
print('                               │')
print('                        M8: Flow Matching (행동)')
print('                               │')
print('                            action → env.step()')
print()
print('M3 완료! 환경이 준비되었으니, M4부터 진짜 AI 아키텍처로 들어갑니다.')